# LDA Emails Full

**Status:** Versão de referência · **Área:** topicos-emails

Notebook autogerado com seções detalhadas, mas falhas salvas de extensões compiladas e tqdm ausente.

Este arquivo está identificado como **Versão de referência**. A organização e a revisão estática não equivalem à validação completa da análise. Consulte as pendências antes de usar seus resultados.

## Como retomar

1. Leia o README desta pasta e suas pendências.
2. Prepare os dados conforme data/README.md quando existir.
3. Reinicie o ambiente e execute em ordem; registre a verificação no README.

[Catálogo](https://github.com/Inocentrax/portfolio-dados/blob/main/CATALOGO.md) · [Plano de continuidade](https://github.com/Inocentrax/portfolio-dados/blob/main/PROXIMOS_PASSOS.md)

## Origem e créditos

Importado de um notebook pessoal do Drive em 15/11/2026. Créditos de cursos, livros e coautores no material original foram preservados. Outputs e metadados pessoais do Colab foram retirados.

---

## Desenvolvimento


# LDA passo‑a‑passo em E‑mails  
*(Notebook autogerado – 2025-06-12 18:11 UTC)*

Este notebook replica – **linha por linha** – o fluxo usado na aula do professor,  
adaptado para o arquivo **`dataset_emails.csv`** (assunto + corpo).  
Siga as células na ordem indicada.  


## 1 — Instalar (ou ajustar) dependências  
> **Execute apenas uma vez.**  
O comando abaixo **não altera** pandas/numpy já presentes no Colab, evitando conflitos.


In [ ]:
# --- Patch para compatibilizar gensim x SciPy >= 2 ---
import types, numpy as np, scipy.linalg
if not hasattr(scipy.linalg, "triu"):
    scipy.linalg.triu = np.triu        # alias
if not hasattr(scipy.linalg, "get_blas_funcs"):
    # SciPy 2 colocou isto em scipy.linalg.blas, então criamos atalhos
    import scipy.linalg.blas as _blas
    scipy.linalg.get_blas_funcs = _blas.get_blas_funcs
# ------------------------------------------------------


In [ ]:
# 1) Remove SciPy 2.x (se presente)
!pip uninstall -y scipy

# 2) Instala versão compatível com gensim
!pip install -q "scipy<2.0,>=1.10"

# 3) Instala gensim e pyLDAvis (sem puxar deps extras)
!pip install -q gensim==4.3.2 pyLDAvis==3.4.1 --no-deps
!pip install -q funcy==1.18


## 2 — Imports e checagem de versões

In [ ]:
import gensim, scipy, pandas as pd, pyLDAvis, spacy
print("SciPy :", scipy.__version__)
print("Gensim:", gensim.__version__)


## 3 — Baixar recursos linguísticos (se ainda não estiverem presentes)

In [ ]:
import subprocess, sys, importlib
import nltk, spacy
nltk.download('punkt')
nltk.download('stopwords')

try:
    nlp = spacy.load('pt_core_news_sm')
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "pt_core_news_sm"])
    nlp = spacy.load('pt_core_news_sm')


## 4 — Carregar CSV de e‑mails

In [ ]:
# Ajuste o caminho se necessário (upload no Colab → painel lateral)
import pandas as pd
data = pd.read_csv('/content/dataset_emails.csv')
print("Colunas no CSV:", list(data.columns)[:10], '...')
# Criar campo texto
data['texto'] = data['assunto'].astype(str) + ' ' + data['corpo'].astype(str)
data[['texto']].head()


## 5 — Pré‑processamento (igual à aula)

In [ ]:
import re, string
from nltk.corpus import stopwords

stop_pt = set(stopwords.words('portuguese'))
allowed_postags = ['NOUN', 'PROPN', 'VERB', 'ADJ', 'ADV']

def preprocess_text(doc):
    doc = re.sub(r'\s+', ' ', doc)
    doc = re.sub(r'http\S+', '', doc)
    doc = doc.lower()

    spacy_doc = nlp(doc)
    tokens = []
    for token in spacy_doc:
        if token.is_stop or token.is_punct or token.text in stop_pt:
            continue
        if token.pos_ not in allowed_postags:
            continue
        lemma = token.lemma_.strip()
        if lemma and lemma not in string.punctuation:
            tokens.append(lemma)
    return tokens


### 5.1 — Tokenizar e lematizar

In [ ]:
tqdm.pandas(desc='Pré‑processando')
data['tokens'] = data['texto'].progress_apply(preprocess_text)
data[['tokens']].head()


## 6 — Construir bigramas/trigramas

In [ ]:
from gensim.models import Phrases
bigram = Phrases(data['tokens'], min_count=5, threshold=10)
bigram_mod = Phraser(bigram)

data['tokens_bigram'] = data['tokens'].apply(bigram_mod)
data[['tokens_bigram']].head()


## 7 — Dicionário e Corpus BoW

In [ ]:
from gensim.corpora import Dictionary
dictionary = Dictionary(data['tokens_bigram'])
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(text) for text in data['tokens_bigram']]
print("Tamanho do vocabulário:", len(dictionary))


## 8 — Treinar LDA (baseline)

In [ ]:
from gensim.models.ldamodel import LdaModel
NUM_TOPICS = 8
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=10,
    chunksize=2000,
    alpha='auto',
    eta='auto'
)

for idx, topic in lda_model.print_topics(-1):
    print(f"Tópico {idx}\n———\n{topic}\n")


## 9 — Encontrar número ótimo de tópicos

In [ ]:
from gensim.models.coherencemodel import CoherenceModel

def compute_coherence_values(start, stop, step=2):
    results = []
    for k in range(start, stop + 1, step):
        model = LdaModel(corpus=corpus, id2word=dictionary,
                         num_topics=k, random_state=42, passes=10, alpha='auto', eta='auto')
        coherence = CoherenceModel(model=model, texts=data['tokens_bigram'],
                                   dictionary=dictionary, coherence='c_v').get_coherence()
        results.append((k, coherence))
        print(f"k={k} → Coerência {coherence:.4f}")
    return results

coherence_values = compute_coherence_values(4, 14, 2)

ks, cvs = zip(*coherence_values)
plt.figure(figsize=(6,4))
plt.plot(ks, cvs, marker='o')
plt.xlabel("Número de tópicos")
plt.ylabel("Coerência c_v")
plt.title("Coerência x Nº de tópicos")
plt.show()


## 10 — Visualização interativa com pyLDAvis

In [ ]:
import pyLDAvis.gensim_models as gensimvis, pyLDAvis
pyLDAvis.enable_notebook()

vis = gensimvis.prepare(lda_model, corpus, dictionary)
vis


## 11 — Salvar modelo (opcional)

In [ ]:
# lda_model.save('lda_emails_final.model')
# dictionary.save('lda_emails.dict')


## Continuidade

Não usar documentação longa como prova de funcionamento; corrigir imports tqdm/plt e Phrases.Phraser, substituir patches e desinstalações ad hoc por ambiente verificado; incorporar só partes úteis.

Registre decisões e resultados verificados no README desta pasta. Evite criar “Untitled” ou “Cópia de” para continuar este trabalho; use o mesmo arquivo e o histórico do Git.